# CDM 2026 — Exploration & Analyse


In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
ROOT = Path('..').resolve()
DATA_RAW = ROOT / 'data' / 'raw'

plt.style.use('dark_background')
sns.set_palette('viridis')

## 1. Chargement des données

In [ ]:
all_path = DATA_RAW / 'all_matches.parquet'
if all_path.exists():
    df = pd.read_parquet(all_path)
    df['date'] = pd.to_datetime(df['date'])
    print(f'{len(df)} matchs chargés')
    display(df.head())
else:
    print('Données manquantes — lance fetch_data.py d\'abord.')

## 2. Distribution des scores

In [ ]:
df_ft = df[df['status'] == 'FT'].copy()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, col, label in zip(axes, ['home_goals', 'away_goals'], ['Buts domicile', 'Buts extérieur']):
    vals = df_ft[col].dropna().astype(int)
    ax.bar(range(vals.max()+1), [( vals == i).sum() for i in range(vals.max()+1)])
    ax.set_title(label)
    ax.set_xlabel('Buts')
    ax.set_ylabel('Matchs')
plt.tight_layout()
plt.show()

print(f"Buts/match domicile (moy): {df_ft['home_goals'].mean():.3f}")
print(f"Buts/match extérieur (moy): {df_ft['away_goals'].mean():.3f}")
print(f"Over 2.5 rate: {(df_ft['home_goals'] + df_ft['away_goals'] > 2.5).mean():.3f}")
print(f"BTTS rate: {((df_ft['home_goals'] > 0) & (df_ft['away_goals'] > 0)).mean():.3f}")

## 3. Modèle Dixon-Coles — prédictions

In [ ]:
from models.dixon_coles import DixonColesModel, MODEL_PATH

if MODEL_PATH.exists():
    model = DixonColesModel.load()
    print('Modèle chargé.')
    
    home, away = 'France', 'Argentina'
    preds = model.predict_outcomes(home, away)
    print(f'\n{home} vs {away}')
    for k, v in preds.items():
        print(f'  {k}: {v:.4f}')
    
    matrix = model.predict_score_matrix(home, away)
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(matrix[:6, :6], annot=True, fmt='.3f', ax=ax,
                xticklabels=range(6), yticklabels=range(6))
    ax.set_xlabel(f'Buts {away}')
    ax.set_ylabel(f'Buts {home}')
    ax.set_title(f'Matrice de score : {home} vs {away}')
    plt.tight_layout()
    plt.show()
else:
    print('Modèle non entraîné. Lance : python models/dixon_coles.py --train')

## 4. Scan value bets — exemple

In [ ]:
from pipeline.value_detector import scan_value_bets, compute_roi_from_log
import yaml

with open(ROOT / 'config.yaml') as f:
    cfg = yaml.safe_load(f)

if MODEL_PATH.exists():
    model = DixonColesModel.load()
    
    # Exemple avec cotes fictives
    test_matches = [
        {'fixture_id': 1, 'date': '2026-06-15', 'home_team': 'France',
         'away_team': 'Mexico', 'venue': 'Estadio Azteca', 'is_friendly': False},
        {'fixture_id': 2, 'date': '2026-06-16', 'home_team': 'Argentina',
         'away_team': 'Brazil', 'venue': 'MetLife Stadium', 'is_friendly': False},
    ]
    test_odds = {
        1: {'1X2': {'home': 2.10, 'draw': 3.40, 'away': 3.20},
            'over_2.5': {'over': 1.90, 'under': 1.90},
            'btts': {'yes': 1.85, 'no': 1.95}},
        2: {'1X2': {'home': 2.50, 'draw': 3.10, 'away': 2.80},
            'over_2.5': {'over': 1.75, 'under': 2.05},
            'btts': {'yes': 1.70, 'no': 2.10}},
    }
    
    value_bets = scan_value_bets(test_matches, model, test_odds, cfg, 200.0)
    print(f'{len(value_bets)} value bet(s) détecté(s)')
    for bet in value_bets:
        print(f"  {bet['home_team']} vs {bet['away_team']} | {bet['market']}/{bet['side']} | Edge: {bet['edge']*100:.1f}%")
else:
    print('Modèle non entraîné.')

## 5. ROI suivi (bets loggés)

In [ ]:
from pipeline.value_detector import compute_roi_from_log, BETS_LOG
import pandas as pd

stats = compute_roi_from_log()
print('ROI courant :', stats)

if BETS_LOG.exists():
    log = pd.read_csv(BETS_LOG)
    display(log.tail(10))